In [ ]:
from pathlib import Path
import pandas as pd

# Read data
DATA_DIR = "results/mnist_8_4/"
data_dir = Path(DATA_DIR)

epoch_data_files = list(data_dir.glob("*.csv"))
inter = []
for f in epoch_data_files:
    data = pd.read_csv(f)
    data["model"] = int(f.stem)
    if len(data) < 5:
        print(len(data))
    data = data[:-5] # removing patience rows
    inter.append(data)

df_epochs = pd.concat(inter, ignore_index=True)




In [ ]:
# Metric normalization according to section 4.4.1
import numpy as np
from scipy.ndimage import gaussian_filter1d 
from sklearn.preprocessing import MinMaxScaler

def normalizer(values):
    values = values.to_numpy(dtype=float)
    values = np.log1p(values)
    values = gaussian_filter1d(values, sigma = 1)
    values = (values - values.min()) / (values.max() - values.min())
    return values

metrics = ["train_loss", "val_loss", "bdm", "entropy"]
df_normalized = df_epochs.sort_values(["model", "epoch"]).copy()
df_normalized[metrics] = df_normalized.groupby("model")[metrics].transform(normalizer)


In [ ]:
print("N models with invalid entropy: ", len(set(df_normalized[df_normalized["entropy"].isna()]["model"])))

In [ ]:
# Manual inspection of invalid entropy models (after normalizing)

df_epochs[df_epochs["model"].isin(set(df_normalized[df_normalized["entropy"].isna()]["model"]))]

In [ ]:
invalid_entropy = set(df_normalized[df_normalized["entropy"].isna()]["model"])
print(invalid_entropy)
results = []

for model, run in df_normalized.groupby("model"):
    row = {"model": model, "n_epochs": len(run)}
    if model in invalid_entropy:
        continue
    loss = run["train_loss"]

    for metric in ["bdm", "entropy"]:
        values = run[metric]

        row[f"{metric}_pearson"] = loss.corr(values, method = "pearson") 
        row[f"{metric}_spearman"] = loss.corr(values, method = "spearman")
    
    results.append(row)

df_results = pd.DataFrame(results)
df_results_n5 = df_results[df_results["n_epochs"] >= 5]
print("Number of valid runs: ", len(df_results_n5))

In [ ]:

sample_means_pears_bdm = []
sample_means_spear_bdm = []
sample_means_pears_entropy = []
sample_means_spear_entropy = []
for i in range(10_000):
    df_bootstrap = df_results_n5.sample(n = len(df_results_n5), replace = True)

    sample_means_pears_bdm.append(df_bootstrap["bdm_pearson"].mean())
    sample_means_spear_bdm.append(df_bootstrap["bdm_spearman"].mean())
    sample_means_pears_entropy.append(df_bootstrap["entropy_pearson"].mean())
    sample_means_spear_entropy.append(df_bootstrap["entropy_spearman"].mean())

print("bdm pearson: ", np.quantile(sample_means_pears_bdm, [0.025, 0.975]))
print("bdm spearman: ", np.quantile(sample_means_spear_bdm, [0.025, 0.975]))
print("entropy pearson: ", np.quantile(sample_means_pears_entropy, [0.025, 0.975]))
print("entropy spearman: ", np.quantile(sample_means_spear_entropy, [0.025, 0.975]))


In [ ]:
median_pears_bdm = np.quantile(sample_means_pears_bdm, 0.5)
median_pears_entropy = np.quantile(sample_means_pears_entropy, 0.5)
print(abs(median_pears_bdm - median_pears_entropy))

median_spear_bdm = np.quantile(sample_means_spear_bdm, 0.5)
median_spear_entropy = np.quantile(sample_means_spear_entropy, 0.5)
print(abs(median_spear_bdm - median_spear_entropy))



In [ ]:
import torch
# Calculating mean model accuracy

model_data_files = list(data_dir.glob("model_*.pt"))
test_accuracies = []

for f in model_data_files:
    data = torch.load(f)
    test_accuracies.append(data["test_accuracy"])

print("Mean accuracy: ", np.mean(test_accuracies))
print("std accuracy: ", np.std(test_accuracies))

In [ ]:

metrics = ["train_loss", "val_loss", "bdm", "entropy"]
boot_results = []

for epoch, runs in df_normalized.groupby("epoch"):
    values = runs[metrics]

    recorded_means = {"train_loss":[], "val_loss":[], "bdm":[], "entropy":[]}

    if len(runs) >= 5:
        for i in range(10_000):
            df_bootstrapped = values.sample(n = len(values), replace = True)
            means = df_bootstrapped[metrics].mean()
            recorded_means["train_loss"].append(means["train_loss"])
            recorded_means["val_loss"].append(means["val_loss"])
            recorded_means["bdm"].append(means["bdm"])
            recorded_means["entropy"].append(means["entropy"])
    
        for metric in metrics:
            boot_results.append({
                "epoch": epoch,
                "metric": metric,
                "mean": runs[metric].mean(),
                "lower": np.quantile(recorded_means[metric], 0.025),
                "upper": np.quantile(recorded_means[metric], 0.975),
                "n_runs": len(runs) / 200
            })
    

df_epoch_boot = pd.DataFrame(boot_results)    

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
label_mapping = {
    "train_loss": "Training Loss",
    "val_loss": "Validation Loss",
    "bdm": "BDM",
    "entropy": "Entropy"            
    }

for metric in metrics:
    data = df_epoch_boot[df_epoch_boot["metric"] == metric]
    line, = ax.plot(data["epoch"], data["mean"], label=label_mapping[metric])
    ax.fill_between(
        data["epoch"],
        data["lower"],
        data["upper"],
        color=line.get_color(),
        alpha=0.2,
    )

ax.plot(df_epoch_boot["epoch"], 
        df_epoch_boot["n_runs"], 
        "--", 
        color="gray", 
        label = "Active Runs")
ax.set(xlabel="Epoch", ylabel="Normalized value")
ax.legend()
ax.margins(x=0)
plt.grid()
plt.show()